In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data = np.genfromtxt(
    "Dry_Bean_Dataset.csv",
    delimiter=",",
    skip_header=1,
    dtype=str
)
rng = np.random.default_rng(seed=42)
rng.shuffle(data)
# Numerical features
X = data[:, :-1].astype(float)

# Class names
labels = data[:, -1]

# Encode classes as integers 0–6
classes, T = np.unique(labels, return_inverse=True)

print("X shape:", X.shape)  # (13611, 16)
print("T shape:", T.shape)  # (13611,)

X shape: (13611, 16)
T shape: (13611,)


In [3]:
T_onehot = np.eye(len(classes))[T]
print(T_onehot.shape)  # (13611, 7)

(13611, 7)


K is the number of classes. Our case K=7

In [4]:
K = T_onehot.shape[1]

In [5]:
print(T[0])
print(T_onehot[0])

3
[0. 0. 0. 1. 0. 0. 0.]


In [6]:
train_x = X[:10500, :]
train_t = T_onehot[:10500,:]
print(train_x.shape)
print(train_t.shape)

(10500, 16)
(10500, 7)


In [7]:
#Data normalization
mean = np.mean(train_x,axis=0)
std = np.std(train_x,axis=0)
train_x = (train_x-mean)/std

We use a second order polynomial basis function

In [8]:
phi = []; #2nd order polynomial estimate
for xn in train_x:
    phi_n = [1]
    for i in range(train_x.shape[1]):
        phi_n.append(xn[i])
    for i in range(train_x.shape[1]):
        for j in range(i,train_x.shape[1]):
            phi_n.append(xn[i]*xn[j])
    phi.append(phi_n)
phi = np.array(phi)
phi.shape

(10500, 153)

In [9]:
W = np.zeros((phi.shape[1],T_onehot.shape[1]))

In [10]:
W[:,0].shape

(153,)


$\mathbf{W}$ means $\{\mathbf{w_1},....,\mathbf{w_K}\}$. $\mathbf{T}$ means $\{\mathbf{t_1},....,\mathbf{t_N}\}$. 
$\mathbf{X}$ means $\{\mathbf{x_1},....,\mathbf{x_N}\}$. $\phi(\mathbf{x_n})$ means the basis function applied to our input vector.   
Training is done using Iteratively reweighted least squares. K here is the number of classes, this case K=7
$$
p(t_k=1\mid \mathbf{x},\mathbf{W})
= y_k
= \frac{\exp\!\left(\mathbf{w}_k^\top\boldsymbol{\phi}(\mathbf{x})\right)}
{\displaystyle\sum_{j=1}^{K}
\exp\!\left(\mathbf{w}_j^\top\boldsymbol{\phi}(\mathbf{x})\right)}
$$

$$
p(\mathbf{t}\mid\mathbf{x},\mathbf{W})
= \prod_{k=1}^{K}
p(t_k=1\mid\mathbf{x},\mathbf{W})^{t_k}
$$

$$
\begin{aligned}
p(\mathbf{T}\mid\mathbf{X},\mathbf{W})
&= \prod_{n=1}^{N}
p(\mathbf{t}_n\mid\mathbf{x}_n,\mathbf{W}) \\
&= \prod_{n=1}^{N}\prod_{k=1}^{K}y_{nk}^{\,t_{nk}},
\end{aligned}
\qquad
y_{nk}
\equiv
\frac{\exp\!\left(\mathbf{w}_k^\top\boldsymbol{\phi}(\mathbf{x}_n)\right)}
{\displaystyle\sum_{j=1}^{K}
\exp\!\left(\mathbf{w}_j^\top\boldsymbol{\phi}(\mathbf{x}_n)\right)}
$$

$$
E(\mathbf{w}_1,\ldots,\mathbf{w}_K)
= -\ln p(\mathbf{T}\mid\mathbf{X},\mathbf{W})
= -\sum_{n=1}^{N}\sum_{k=1}^{K}t_{nk}\ln y_{nk}
$$

$$
\nabla_{\mathbf{w}_j}E(\mathbf{w}_1,\ldots,\mathbf{w}_K)
= \sum_{n=1}^{N}(y_{nj}-t_{nj})
\boldsymbol{\phi}(\mathbf{x}_n)
$$

$$
\nabla_{\mathbf{w}_k}\nabla_{\mathbf{w}_j}
E(\mathbf{w}_1,\ldots,\mathbf{w}_K)
= \sum_{n=1}^{N}
y_{nk}(\delta_{kj}-y_{nj})
\boldsymbol{\phi}(\mathbf{x}_n)
\boldsymbol{\phi}(\mathbf{x}_n)^\top
$$

$$
\mathbf{w}^{(\mathrm{new})}
= \mathbf{w}^{(\mathrm{old})}
- \mathbf{H}^{-1}\nabla_{\mathbf{w}}E
$$

$$
\nabla_{\mathbf{w}}E
=
\begin{pmatrix}
\nabla_{\mathbf{w}_1}E \\
\vdots \\
\nabla_{\mathbf{w}_K}E
\end{pmatrix},
\qquad
\nabla_{\mathbf{w}}E\in\mathbb{R}^{KD}
$$

The gradient here is a column stacked vector of each invidual weight. The hessian is a K*K Block matrix where each block (j,k) is given by the double gradient equation above. We use the moore penrose inverse instead of the true inverse for the hessian for numerical stability. The last class is kept as a reference class i.e the weights are fixed at zero to prevent singular hessian

In [11]:
iterations = 0
while(iterations < 10):
    GRAD = []
    for j in range(K):
        grad = np.zeros((phi.shape[1],1))
        for n in range(phi.shape[0]):
            phi_xn = phi[n].reshape((phi.shape[1],1))
            activations = W.T@phi_xn
            activations-=np.max(activations)
            denominator = np.sum(np.exp(activations))
            ynj = np.exp(activations[j])
            ynj/=denominator
            tnj = train_t[n][j]
            grad+= (ynj - tnj)*phi_xn
        GRAD.append(grad)
    GRAD = np.vstack(GRAD)
    H = []
    for j in range(K):
        for k in range(K):
            hjk = np.zeros((phi.shape[1],phi.shape[1]))
            for n in range(phi.shape[0]):
                denominator = 0
                phi_xn = phi[n].reshape((phi.shape[1],1))
                activations = W.T@phi_xn
                activations-=np.max(activations)
                denominator = np.sum(np.exp(activations))
                ynj = np.exp(activations[j])/denominator
                ynk = np.exp(activations[k])/denominator
                if(k == j):
                    hjk+= ynk*(1-ynk)*phi_xn@phi_xn.T
                else:
                    hjk+=-ynk*ynj*phi_xn@phi_xn.T
            H.append(hjk)
        
    D = phi.shape[1]
    C = K - 1                 # Last class is the reference class
    

    rows_h2 = [H[i*K:(i+1)*(K)]for i in range(K)]
    H2 = np.block(rows_h2)
    H2 = H2[:C*D,:C*D]
    
    # Gradients for the trainable classes
    g = GRAD[:C * D].reshape(C * D, 1)
    
    # Stack the corresponding columns of W vertically
    w_old = W[:, :C].T.reshape(C * D, 1)
    
    # Newton / IRLS update: w_new = w_old - H2^{-1} g
    update = 0
    update = np.linalg.pinv(H2,hermitian=True,rcond=1e-12)@g #hermitian=True means Hessian is symmetric
    w_new = w_old - 0.5*update
    
    # Put each updated class vector back into its column
    W[:, :C] = w_new.reshape(C, D).T
    W[:, -1] = 0.0
    
    iterations += 1
    print(f"{iterations} completed")

                
            
            
        
        

1 completed
2 completed
3 completed
4 completed
5 completed
6 completed
7 completed
8 completed
9 completed
10 completed


In [12]:
test_x = X[10500:, :]
test_t = T_onehot[10500:,:]
test_x = (test_x-mean)/std
print(test_x.shape)
print(test_t.shape)

(3111, 16)
(3111, 7)


Apply the basis function to our test data

In [13]:
phi_test = []; 
for xn in test_x:
    phi_n = [1]
    for i in range(test_x.shape[1]):
        phi_n.append(xn[i])
    for i in range(test_x.shape[1]):
        for j in range(i,test_x.shape[1]):
            phi_n.append(xn[i]*xn[j])
    phi_test.append(phi_n)
phi_test = np.array(phi_test)
phi_test.shape

(3111, 153)

In [14]:
total_patterns = phi_test.shape[0]
misclassified = 0
for n in range(phi_test.shape[0]):
    activations = W.T@phi_test[n]
    activations-=np.max(activations)
    denominator = np.sum(np.exp(activations))
    prob = np.exp(activations)
    prob/=denominator
    pred = np.argmax(prob)
    actual = np.argmax(test_t[n])
    if(pred!=actual):
        misclassified+=1
print(f"Error: {(misclassified/total_patterns)*100}%")

Error: 7.971713275474124%


In [15]:
Wmap = np.zeros((phi.shape[1],T_onehot.shape[1]))

In [16]:
W[:,0].shape

(153,)

In [17]:
Sn = 0

Now I am going to define a prior over the weights. I assume that each weight is independent and identitcally distributed
$$
p(\mathbf{T}\mid\mathbf{W})
= \prod_{n=1}^{N}\prod_{k=1}^{K}y_{nk}^{\,t_{nk}}
$$

$$
\mathbf{W}=\{\mathbf{w}_1,\ldots,\mathbf{w}_K\}
$$

$$
p(\mathbf{W})=\prod_{k=1}^{K}p(\mathbf{w}_k)
$$

$$
p(\mathbf{w}_k)
=\mathcal{N}\!\left(\mathbf{w}_k\mid\mathbf{m}_{k0},\mathbf{S}_{k0}\right)
$$

$$
p(\mathbf{W}\mid\mathbf{T})
\propto p(\mathbf{T}\mid\mathbf{W})p(\mathbf{W})
$$

$$
p(\mathbf{W}\mid\mathbf{T})
\propto
\left(\prod_{n=1}^{N}\prod_{k=1}^{K}y_{nk}^{\,t_{nk}}\right)
\left(\prod_{k=1}^{K}
\mathcal{N}\!\left(\mathbf{w}_k\mid\mathbf{m}_{k0},\mathbf{S}_{k0}\right)\right)
$$

$$
p(\mathbf{W}\mid\mathbf{T})
\propto
\left(\prod_{n=1}^{N}\prod_{k=1}^{K}y_{nk}^{\,t_{nk}}\right)
\exp\!\left[
-\frac{1}{2}\sum_{k=1}^{K}
(\mathbf{w}_k-\mathbf{m}_{k0})^\top
\mathbf{S}_{k0}^{-1}
(\mathbf{w}_k-\mathbf{m}_{k0})
\right]
$$

$$
\begin{aligned}
-\ln p(\mathbf{W}\mid\mathbf{T})
&= -\sum_{n=1}^{N}\sum_{k=1}^{K}t_{nk}\ln y_{nk} \\
&\quad +\frac{1}{2}\sum_{k=1}^{K}
(\mathbf{w}_k-\mathbf{m}_{k0})^\top
\mathbf{S}_{k0}^{-1}
(\mathbf{w}_k-\mathbf{m}_{k0})
\end{aligned}
$$

$$
-\nabla_{\mathbf{w}_j}\ln p(\mathbf{W}\mid\mathbf{T})
=
\sum_{n=1}^{N}(y_{nj}-t_{nj})\boldsymbol{\phi}(\mathbf{x}_n)
+\mathbf{S}_{j0}^{-1}(\mathbf{w}_j-\mathbf{m}_{j0})
$$

$$
\begin{aligned}
-\nabla_{\mathbf{w}_k}\nabla_{\mathbf{w}_j}
\ln p(\mathbf{W}\mid\mathbf{T})
&= \sum_{n=1}^{N}
y_{nk}(\delta_{kj}-y_{nj})
\boldsymbol{\phi}(\mathbf{x}_n)
\boldsymbol{\phi}(\mathbf{x}_n)^\top \\
&\quad + \delta_{kj}\mathbf{S}_{j0}^{-1}
\end{aligned}
$$

I am choosing the gaussians to have zero mean and covariance $\alpha^{-1}I$

$$
\mathbf{m}_{k0}=\mathbf{0}
\qquad \text{for all } k
$$

$$
\mathbf{S}_{k0}=\alpha^{-1}\mathbf{I}
\qquad \text{for all } k
$$

I am going to now approximate the posterior using Laplace Approximation, which gives a gaussian that is centered at the mode of the posterior distribution $\mathbf{w_\text{MAP}}$ and covariance matrix $\mathbf{S_N}$ which is the inverse of the hessian of the negative log posterior evaluated at $\mathbf{w_\text{MAP}}$ . $\mathbf{w_\text{MAP}}$ can be found using IRLS like before

$$
q(\mathbf{w})
= \mathcal{N}\!\left(
\mathbf{w}\mid\mathbf{w}_{\mathrm{MAP}},\mathbf{S}_N
\right)
$$

$$
\mathbf{w}_{\mathrm{MAP}}
=
\begin{pmatrix}
\mathbf{w}_{1,\mathrm{MAP}} \\
\vdots \\
\mathbf{w}_{K,\mathrm{MAP}}
\end{pmatrix},
\qquad
\mathbf{w}
=
\begin{pmatrix}
\mathbf{w}_1 \\
\vdots \\
\mathbf{w}_K
\end{pmatrix}
$$

In [18]:
iterations = 0
alpha = 1
while(iterations < 30):
    GRAD = []
    for j in range(K):
        grad = np.zeros((phi.shape[1],1))
        for n in range(phi.shape[0]):
            phi_xn = phi[n].reshape((phi.shape[1],1))
            activations = Wmap.T@phi_xn
            activations-=np.max(activations)
            denominator = np.sum(np.exp(activations))
            ynj = np.exp(activations[j])
            ynj/=denominator
            tnj = train_t[n][j]
            grad+= (ynj - tnj)*phi_xn
        grad+= Wmap[:,j].reshape((Wmap[:,j].shape[0],1))*alpha
        GRAD.append(grad)
    GRAD = np.vstack(GRAD)
    H = []
    for j in range(K):
        for k in range(K):
            hjk = np.zeros((phi.shape[1],phi.shape[1]))
            for n in range(phi.shape[0]):
                denominator = 0
                phi_xn = phi[n].reshape((phi.shape[1],1))
                activations = Wmap.T@phi_xn
                activations-=np.max(activations)
                denominator = np.sum(np.exp(activations))
                ynj = np.exp(activations[j])/denominator
                ynk = np.exp(activations[k])/denominator
                if(k == j):
                    hjk+= ynk*(1-ynk)*phi_xn@phi_xn.T
                else:
                    hjk+=-ynk*ynj*phi_xn@phi_xn.T
            if(k == j):
                hjk+=alpha*np.identity(phi.shape[1])
            
            H.append(hjk)
    D = phi.shape[1]
    C = K 
    
    rows_h2 = [H[i*K:(i+1)*(K)]for i in range(K)]
    H2 = np.block(rows_h2)
    # Gradients for the trainable classes
    g = GRAD[:C * D].reshape(C * D, 1)
    
    # Stack the corresponding columns of W vertically
    w_old = Wmap[:, :C].T.reshape(C * D, 1)
    
    # Newton / IRLS update: w_new = w_old - H2^{-1} g
    if np.max(np.abs(g)) < 1e-6: #I put a convergence criteron. If the maximum norm of all gradients is too small, Im halting training
        break
    update = np.linalg.pinv(H2,hermitian=True,rcond=1e-12)@g 
    w_new = w_old - 0.5*update
    
    # Put each updated class vector back into its column
    Wmap[:, :C] = w_new.reshape(C, D).T
    
    iterations += 1
    print(f"{iterations} completed")

                
            
            
        
        

1 completed
2 completed
3 completed
4 completed
5 completed
6 completed
7 completed
8 completed
9 completed
10 completed
11 completed
12 completed
13 completed
14 completed
15 completed
16 completed
17 completed
18 completed
19 completed
20 completed
21 completed
22 completed
23 completed
24 completed
25 completed
26 completed
27 completed
28 completed
29 completed
30 completed


In [19]:

H = []
for j in range(K):
    for k in range(K):
        hjk = np.zeros((phi.shape[1],phi.shape[1]))
        for n in range(phi.shape[0]):
            denominator = 0
            phi_xn = phi[n].reshape((phi.shape[1],1))
            activations = Wmap.T@phi_xn
            activations-=np.max(activations)
            denominator = np.sum(np.exp(activations))
            ynj = np.exp(activations[j])/denominator
            ynk = np.exp(activations[k])/denominator
            if(k == j):
                hjk+= ynk*(1-ynk)*phi_xn@phi_xn.T
            else:
                hjk+=-ynk*ynj*phi_xn@phi_xn.T
        if(k == j):
            hjk+=alpha*np.identity(phi.shape[1])
        
        H.append(hjk)
    
D = phi.shape[1]
C = K 

rows_h2 = [H[i*K:(i+1)*(K)]for i in range(K)]
H2 = np.block(rows_h2)
Sn = np.linalg.inv(H2)

In [20]:
total_patterns = phi_test.shape[0]
misclassified = 0
for n in range(phi_test.shape[0]):
    activations = Wmap.T@phi_test[n]
    activations-=np.max(activations)
    denominator = np.sum(np.exp(activations))
    prob = np.exp(activations)
    prob/=denominator
    pred = np.argmax(prob)
    actual = np.argmax(test_t[n])
    if(pred!=actual):
        misclassified+=1
print(f"Error: {(misclassified/total_patterns)*100}%")

Error: 7.650273224043716%


Now I am going to train a generative classifier. Im going to model the class conditional densities as a gaussian with shared covariance matrix
$$
p(C_k)=\pi_k,
\qquad
\sum_{k=1}^{K}\pi_k=1
$$

$$
p(\mathbf{x}\mid C_k)
=\mathcal{N}\!\left(\mathbf{x}\mid\boldsymbol{\mu}_k,\boldsymbol{\Sigma}\right)
$$

$$
p(\mathbf{t}_n,\mathbf{x}_n)
=\prod_{k=1}^{K}
\left[p(\mathbf{x}_n\mid C_k)p(C_k)\right]^{t_{nk}}
$$

$$
\begin{aligned}
p(\mathcal{D}\mid\boldsymbol{\pi},\boldsymbol{\mu},\boldsymbol{\Sigma})
&=\prod_{n=1}^{N}p(\mathbf{x}_n,\mathbf{t}_n) \\
&=\prod_{n=1}^{N}\prod_{k=1}^{K}
\left[
\mathcal{N}\!\left(
\mathbf{x}_n\mid\boldsymbol{\mu}_k,\boldsymbol{\Sigma}
\right)\pi_k
\right]^{t_{nk}}
\end{aligned}
$$

$$
\boldsymbol{\pi}=\{\pi_1,\ldots,\pi_K\}
$$

$$
\boldsymbol{\mu}
=\{\boldsymbol{\mu}_1,\ldots,\boldsymbol{\mu}_K\}
$$
Here $\mathcal{D}$ means the entire dataset comprising of input vectors and training labels. Using maximum likelihood we get:
$$
N_k=\sum_{n=1}^{N}t_{nk}
\qquad
\text{(number of patterns in class } k\text{)}
$$

$$
\pi_k=\frac{N_k}{N}
$$

$$
\boldsymbol{\mu}_k
=\frac{1}{N_k}\sum_{n=1}^{N}t_{nk}\mathbf{x}_n
$$

$$
\boldsymbol{\Sigma}
=\frac{1}{N}\sum_{n=1}^{N}\sum_{k=1}^{K}
t_{nk}
(\mathbf{x}_n-\boldsymbol{\mu}_k)
(\mathbf{x}_n-\boldsymbol{\mu}_k)^\top
$$

In [21]:
#Generative classifier
pi = np.zeros((K,))
for n in range(train_t.shape[0]):
    pi[train_t[n].argmax()]+=1

mu = []
for k in range(K):
    mu_k = np.zeros((train_x.shape[1],1))
    for n in range(train_x.shape[0]):
        if(np.argmax(train_t[n]) == k):
            mu_k+=train_x[n].reshape((train_x[n].shape[0],1))
    mu_k/=pi[k]
    mu.append(mu_k)
sigma = np.zeros((train_x.shape[1],train_x.shape[1]))
for n in range(train_x.shape[0]):
    for k in range(K):
        if(np.argmax(train_t[n]) == k):
            xn = train_x[n].reshape((train_x.shape[1],1))
            mu_k = mu[k]
            sigma+=(xn-mu_k)@((xn-mu_k).T)
sigma/=train_t.shape[0]
pi/=train_t.shape[0]

In [22]:
def gaussian(mu,cov_matrix,x):
    C = (np.linalg.det(cov_matrix)**(-0.5))*1/((2*np.pi)**(x.shape[0]/2))
    return C*np.exp(-0.5*((x-mu).T@np.linalg.inv(cov_matrix)@(x-mu)))

Now we are going to predict using bayes theorem
$$
p(C_k\mid\mathbf{x}_n)
=
\frac{p(\mathbf{x}_n\mid C_k)\,p(C_k)}
{\displaystyle\sum_{j=1}^{K}
p(\mathbf{x}_n\mid C_j)\,p(C_j)}
$$


In [23]:
total_patterns = test_x.shape[0]
misclassified = 0
for n in range(test_x.shape[0]):
    test_xn = test_x[n].reshape((test_x.shape[1],1))
    denominator = 0
    for k in range(K):
        denominator+=gaussian(mu[k],sigma,test_xn)*pi[k]
    probabilities = np.zeros((K,))
    for k in range(K):
        probabilities[k] = (gaussian(mu[k],sigma,test_xn)*pi[k])/denominator
    pred = np.argmax(probabilities)
    if(pred != np.argmax(test_t[n])):
        misclassified+=1

print(f"Error: {(misclassified/total_patterns)*100} %")

Error: 10.028929604628736 %


/tmp/ipykernel_28809/3100919790.py:10: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  probabilities[k] = (gaussian(mu[k],sigma,test_xn)*pi[k])/denominator
